In [ ]:
#Célula 1
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, regexp_replace, sum, avg, count, desc, round, substring 
import matplotlib.pyplot as plt
import unicodedata
import re

In [ ]:
#Célula 2
spark = SparkSession.builder\
.appName("Bolsa Familia")\
.config("spark.driver.memory", "8g") \
.config("spark.executor.memory", "8g") \
.config("spark.sql.shuffle.partitions", "50")\
.getOrCreate()

#Caminho do Arquivo

In [ ]:
#Célula 3

caminho_csv ="../dados/pagamentos.csv"

df = spark.read\
.option ("header", True)\
.option ("inferSchema", True)\
.option ("sep", ";")\
.option ("encoding","ISO-8859-1")\
.csv(caminho_csv)

In [ ]:
#Célula 4

df.show(5)
df.printSchema()

In [ ]:
#Célula 5

df_tratado = df

colunas_padrao = {
    "MÊS COMPETÊNCIA": "data_competencia",
    "MÊS REFERÊNCIA": "data_referencia",
    "CÓDIGO MUNICÍPIO SIAFI": "codigo_municipio",
    "UF": "uf",
    "NOME MUNICÍPIO": "nome_municipio",
    "CPF FAVORECIDO": "cpf_favorecido",
    "NIS FAVORECIDO" : "nis_favorecido",
    "NOME FAVORECIDO": "nome_favorecido",
    "VALOR PARCELA": "valor_parcela"
}

for antiga, nova in colunas_padrao.items():
    df_tratado = df_tratado.withColumnRenamed(antiga,nova)



In [ ]:
#Célula 6
df_tratado.show(1)


In [ ]:
#Célula 7

df_tratado = (
    df_tratado
    .dropna()
    .withColumn(
        "valor_parcela", 
        regexp_replace(col("valor_parcela"), ",", ".").cast("decimal(10,2)")
    )
    .withColumn(
        "ano_competencia", 
        substring(col("data_competencia"), 1, 4)
    )
    .withColumn(
        "mês_competencia", 
        substring(col("data_competencia"), 5, 2)
    )
)

In [ ]:
#Célula 8

df_tratado.printSchema()


In [ ]:
#Célula 9

df_tratado.agg(
    round(sum("valor_parcela"), 2).alias("total_pago"),
    round(avg("valor_parcela"), 2).alias("media_pagamento")
).show()

In [ ]:
#Célula 10

df_tratado.groupBy("uf")\
    .agg(sum("valor_parcela").alias("total_pago"))\
    .orderBy("total_pago" , ascending=False)\
    .show(5)

In [ ]:
#Célula 11

ranking_favorecido = df_tratado.groupBy("cpf_favorecido" , "nome_favorecido")\
    .agg(
        sum("valor_parcela").alias("valor_total_acumulado"),
        count("valor_parcela").alias("quantidade_parcelas")
    )\
    .orderBy(desc("valor_total_acumulado"))

In [ ]:
#Célula 12
print("Rankink dos 10 primeiros com valores acumulados")
ranking_favorecido.show(5, truncate=False)

In [ ]:
#Célula 13

media_geral = df_tratado.agg(avg("valor_parcela")).collect()[0][0]
print(f"Para fins de comparação, media geral de cada parcela e R$ {media_geral:.2f}")

In [ ]:
df_uf = df_tratado.groupBy("uf") \
    .agg(sum("valor_parcela").alias("total_pago")) \
    .orderBy("total_pago", ascending=False) \
    .limit(10) \
    .toPandas()

# Gráfico
plt.figure()
plt.bar(df_uf["uf"], df_uf["total_pago"])
plt.title("Top 10 UFs - Total Pago Bolsa Família")
plt.xlabel("UF")
plt.ylabel("Total Pago")
plt.show()